# Module 6.1 — Photos and Adoption

**Module 6: Vision + Deployment | Week 1 | Lecture (Session 6.1)**

*Source: Refactored from archived `M05-4_photos_and_adoption.ipynb` (Session 28, 2026-05-15). Cell 12's 89-line monolith split into 4 cells; ROC-AUC added alongside accuracy; backbone upgraded to ConvNeXt-Tiny; bundle ships multi-photo–averaged features (one feature vector per dog, averaged across all available photos in the listing). Session 35 (2026-05-21): SHARED_6 feature set replaces OVERLAP_5; all-dogs training with zero-padded CNN for photless dogs.*

---

In M05 you hit a wall. With five hand-crafted features and a careful cascade, you got the data up to about 0.62 accuracy on *adopted in 30 days*. More features bought you a couple more points. More model bought you nothing. The wall is in the features, not the model.

Today we try to break that wall with a different *kind* of feature — letting a pretrained vision model turn each dog's photo into a usable feature vector. You'll see what a modern CNN does to a photo, how to extract that vector, and how much it adds to a stronger tabular baseline.

By the end of this lecture you should be able to say, with evidence:

- A pretrained ConvNeXt-Tiny turns each photo into 768 numbers; those numbers carry real adoption signal.
- On the full PetFinder dog set, image-only CNN features and the 6-feature SHARED set reach similar AUC (~0.67 vs ~0.66). With good demographic features, tabular carries comparable signal to 768 CNN features — a much closer race than M05's 5-feature OVERLAP set would suggest.
- The combined model — tabular + image — pulls clearly ahead at ~0.64 accuracy / ~0.70 AUC, about +0.03 AUC over either source alone. The two sources carry complementary information: the CNN sees composition and visual quality; the tabular features encode demographics the CNN can't reliably extract on its own.
- Photos are a *more* effective feature source if you use *all* the photos a listing has, not just the first one. PetFinder dogs have 3–4 photos on average; averaging the CNN features across them is the cheapest way to roughly double the photo lift.
- Including every dog in training — even the 194 with no photos (zero-padded CNN features) — lifts performance. A more diverse training set helps even when some inputs are empty.

## Glossary

New vocabulary in this lecture. Most of it carries forward into the rest of M06.

- **SHARED_6** — the 6-feature demographic set used throughout M06: `age_adult`, `is_young`, `is_mixed_breed`, `gender_male`, `gender_female`, `sterilized_yes`. Drops `size` from M05's OVERLAP set (the CNN encodes body size from photos) and adds the two biggest M05 lifts (`is_young` and `is_mixed_breed`). Same features deploy to Taiwan in 6.5–6.6.
- **Pretrained model** — a neural network that has already been trained on a large dataset and is now reused as-is. Saves you the (huge) cost of training from scratch.
- **ImageNet** — the standard pretrain corpus for vision: ~1.2M photos across 1,000 categories (cats, dogs, hot dogs, electric eels, etc.). Most pretrained CNNs you'll see were trained on it.
- **ConvNeXt-Tiny** — a 2022 CNN architecture inspired by vision transformers but built as a pure-conv stack. About 28M parameters; produces 768-dim feature vectors from its penultimate layer. We use it because it gives a noticeably stronger photo signal on this dataset than older networks like ResNet-18.
- **ResNet** — the older CNN family (2015) that introduced residual connections. M04 taught you how ResNets work; ConvNeXt is "ResNet ideas, modernized." Either gives you image features; ConvNeXt's are sharper on most natural-image tasks.
- **Penultimate layer** — the second-to-last layer of a classification network. The last layer maps to class scores; the penultimate layer holds the **semantic features** the model used to make that decision. That's what we want.
- **Identity head** — the trick we use to pull penultimate features out of a pretrained model: replace the final classification layer with a no-op (`nn.Identity()`) so the model's output *is* the penultimate features. For models with multi-layer classifier heads (like ConvNeXt), we drop only the final linear and keep the layer-norm + flatten that produce the feature vector.
- **Feature vector** — a fixed-length list of numbers that describes one example. The 6-feature SHARED set is a feature vector of length 6. CNN photo features here are a feature vector of length 768.
- **CNN features** — shorthand for "the feature vector a CNN produces for an image." Used interchangeably with *image features* and *photo features* in this lecture.
- **AUC** (Area Under the ROC Curve) — a ranking metric. Instead of asking "did the model put this example on the right side of 0.5?", AUC asks "if I pick one positive example and one negative example at random, how often does the model assign the positive one a higher probability?" 1.0 is perfect ranking; 0.5 is random. Sensitive to small distributional shifts that accuracy at threshold 0.5 misses.

## Why photos matter

A few things to take as given (most are well-documented in shelter literature):

- A clear, well-lit, face-visible photo gets more clicks on adoption listings.
- Photos with humans in them — especially gentle-looking interactions — perform well.
- Bad photos (dark, blurry, far away, behind kennel bars) actively hurt.
- The number of photos matters too: zero photos is a death sentence; 4–6 is roughly optimal.

The shelter you're working with probably already knows this in some intuitive form. The question is whether we can quantify it from data — and eventually use it to suggest specific things for *your* dog's listing.

## Two approaches to image features

A model needs numbers. A photo is millions of pixels. We need a way to compress "this picture of a dog" into something the model can work with.

**Approach A: hand-engineered features.** Write code that measures specific properties — brightness, contrast, sharpness, dominant color, whether there's a face. Each property becomes a column. This works for the obvious stuff but misses everything subtle ("is the dog in a friendly pose?", "is the lighting cinematic or fluorescent?"). You only get features you thought to write.

**Approach B: pretrained CNN features.** Take a CNN that was already trained on millions of natural images (M04 territory) and grab the activations from its penultimate layer. That gives you a single vector — usually 512 or 2048 numbers — per photo. The CNN already learned features that are useful across many vision tasks; we don't need it to *classify* dog photos, just to compress them into something rich.

We'll use Approach B. The trade-off is that the features are uninterpretable — you can't easily say "feature #237 means friendly eyes" — but you don't have to know what to measure.

## Setup — load data and bundles

M06 uses two data bundles:

- **`petfinder-mini.zip`** — same bundle you've used since M05. Ships the dog metadata (`train.csv`), the 300-photo sample (`train_images_sample/`), and the M05 toolkit (`m05_labbench.py`).
- **`m06-photos.zip`** — new for M06. Ships `petfinder_cnn_features.csv` and a `my_dog.jpg` template. The CSV holds **one feature vector per dog, averaged across all the photos that listing had** (3–4 on average, up to 30). We'll see why "all photos" beats "just the first photo" further down.

Live feature extraction from photos works fine, but on free Colab it's slow enough to interrupt the lecture (~30 min for the full sample, several hours for multi-photo). Instead, we live-extract a few sample dogs from one photo each to show the pipeline, then load the pre-extracted multi-photo CSV for the actual model comparison.

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

In [ ]:
PETFINDER_URL  = 'CONFIGURE_ME'  # ← your instructor will give you the M05 bundle URL
M06_PHOTOS_URL = 'CONFIGURE_ME'  # ← your instructor will give you the M06 bundle URL
DATA_DIR = '/content/petfinder'

os.makedirs(DATA_DIR, exist_ok=True)

def fetch_bundle(url, marker_file, label):
    """Download + unzip a bundle if its marker file isn't already on disk."""
    if os.path.exists(f'{DATA_DIR}/{marker_file}'):
        return
    if url == 'CONFIGURE_ME':
        raise RuntimeError(f'Set {label} above to your instructor URL.')
    import gdown
    zip_path = f'{DATA_DIR}/_bundle.zip'
    gdown.download(url=url, output=zip_path, quiet=False, fuzzy=True)
    if not os.path.exists(zip_path):
        raise RuntimeError(f'{label} download failed — check URL and Drive share settings.')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)
    if not os.path.exists(f'{DATA_DIR}/{marker_file}'):
        raise RuntimeError(f'{marker_file} not found after extraction — wrong zip?')

fetch_bundle(PETFINDER_URL,  'train.csv',                   'PETFINDER_URL')
fetch_bundle(M06_PHOTOS_URL, 'petfinder_cnn_features.csv', 'M06_PHOTOS_URL')
print(f'Bundles ready in {DATA_DIR}')


In [ ]:
df = pd.read_csv(f'{DATA_DIR}/train.csv')
dogs = df[df['Type'] == 1].copy()
dogs['adopted_in_30_days'] = (dogs['AdoptionSpeed'] <= 2).astype(int)
print(f'Dogs in dataset: {len(dogs):,}')
print(f'Adopted in 30 days (base rate): {dogs["adopted_in_30_days"].mean():.3f}')

## Photo count itself is a feature

Before we throw a pretrained CNN at the photos, check the simplest possible photo feature: **how many photos does the listing have?** Shelter literature claims a clean pattern — zero photos hurts, 4-6 is optimal. PetFinder ships this directly as `PhotoAmt`. Worth one minute of analysis before committing to anything more complex.

In [ ]:
buckets = [
    ('PhotoAmt = 0',    dogs['PhotoAmt'] == 0),
    ('PhotoAmt = 1',    dogs['PhotoAmt'] == 1),
    ('PhotoAmt = 2-3',  dogs['PhotoAmt'].between(2, 3)),
    ('PhotoAmt = 4-6',  dogs['PhotoAmt'].between(4, 6)),
    ('PhotoAmt = 7-10', dogs['PhotoAmt'].between(7, 10)),
    ('PhotoAmt >= 11',  dogs['PhotoAmt'] >= 11),
]
print(f"{'bucket':18s}  {'n':>5s}  {'30d rate':>9s}  {'90d rate':>9s}  {'never':>7s}")
for label, mask in buckets:
    s = dogs[mask]
    if len(s) < 5:
        continue
    never_rate = (s['AdoptionSpeed'] == 4).astype(int).mean()
    rate_90 = (s['AdoptionSpeed'] <= 3).astype(int).mean()
    print(f"{label:18s}  {len(s):>5d}  {s['adopted_in_30_days'].mean():>9.3f}  "
          f"{rate_90:>9.3f}  {never_rate:>7.3f}")

Three things worth noticing in that table:

- **Zero photos is severe.** ~59% of zero-photo listings never adopt within the study window, vs ~30% for any positive count. The shelter-literature claim holds.
- **The 30-day rate peaks at 4-6 photos and drops for 7+.** But the 90-day rate keeps climbing — listings with 7-10 photos hit the highest 90-day rate (~0.82) and the lowest never-adopted rate (~0.18).
- **Many photos may signal a harder dog.** Rescuers who upload more photos are probably working harder to promote a dog that's harder to place (special needs, older, behavioral notes). Those dogs adopt slower in the short term — but more reliably eventually.

So `PhotoAmt` carries real signal, and that signal is non-linear: a single coefficient on `PhotoAmt` in a linear model would miss the 30d-vs-90d divergence. We won't add `PhotoAmt` to the comparison below — the CNN features end up encoding most of what `PhotoAmt` captures plus the photo *content*. But this is the kind of analysis you do first: simple-feature sanity check before complex-feature commitment. The deep dive `M06-deep_photo_dosage.ipynb` explores the dose-response curve in more detail.

## The pretrained CNN

We load **ConvNeXt-Tiny** with the `IMAGENET1K_V1` weights (~110 MB). ConvNeXt's classification head is `Sequential(LayerNorm2d, Flatten, Linear)` — we drop only the final `Linear` so the model returns the 768-dim feature vector produced by `LayerNorm + Flatten`. All parameters are frozen.

Each pretrained-weights bundle ships its own preprocessing function. We use `weights.transforms()` to apply exactly what ConvNeXt-Tiny was trained with: resize, center-crop, normalize with ImageNet mean/std. Skip these and the features will be garbage — the network expects inputs in a specific range.

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
backbone = models.convnext_tiny(weights=weights)

# ConvNeXt's classifier = Sequential(LayerNorm2d, Flatten, Linear).
# Drop only the final Linear so the model returns the 768-dim feature vector
# (LayerNorm2d + Flatten still run).
backbone.classifier = nn.Sequential(backbone.classifier[0], backbone.classifier[1])
backbone.eval()
backbone = backbone.to(device)
for p in backbone.parameters():
    p.requires_grad = False

# Each weights bundle ships its own preprocessing. Use it as-is.
preprocess = weights.transforms()

print(f'Pretrained ConvNeXt-Tiny ready on {device}.')
print(f'Preprocessing: {preprocess}')

## Demo 1 — extract features from one photo

Open a single photo, run it through the pipeline, look at the resulting 768-number vector. This is the entire mechanism in one cell.

In [ ]:
photo_dir = f'{DATA_DIR}/train_images_sample'
photo_files = sorted(f for f in os.listdir(photo_dir) if f.endswith('-1.jpg'))

sample_path = f'{photo_dir}/{photo_files[0]}'
sample_img = Image.open(sample_path).convert('RGB')

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(sample_img)
ax.set_title(photo_files[0])
ax.axis('off')
plt.show()

tensor = preprocess(sample_img).unsqueeze(0).to(device)
with torch.no_grad():
    features = backbone(tensor).cpu().numpy().squeeze()

print(f'Feature vector shape: {features.shape}')
print(f'First 8 values: {features[:8].round(3)}')
print(f'Range: {features.min():.2f} to {features.max():.2f}')

Each photo becomes 768 numbers. We can't easily say what each number means — but the model trained on millions of images thinks they're a useful summary of what's in the picture.

## Demo 2 — extract features from the 300 sample dogs

Same code as above, in a loop. On the bundled 300-photo sample this runs in seconds on GPU, a minute or so on CPU. (In the lab you'll do the same for your own dog's photo.)

In [ ]:
def extract_features(image_path):
    img = Image.open(image_path).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        return backbone(tensor).cpu().numpy().squeeze()

demo_features = {}
for filename in photo_files:
    pet_id = filename.replace('-1.jpg', '')
    demo_features[pet_id] = extract_features(f'{photo_dir}/{filename}')

demo_df = pd.DataFrame.from_dict(
    demo_features, orient='index',
    columns=[f'img_{i}' for i in range(768)],
)
demo_df.index.name = 'PetID'
print(f'Live-extracted CNN features for {len(demo_df)} dogs. Shape: {demo_df.shape}')

## Do photos help predict adoption?

Three models, same target, different feature sets:

- **Tabular-only:** the 6-feature SHARED set (`age_adult`, `is_young`, `is_mixed_breed`, `gender_male`, `gender_female`, `sterilized_yes`). This replaces M05's 5-feature OVERLAP — we dropped `size` (the CNN sees body size in photos) and added `is_young` and `is_mixed_breed` (M05's two biggest single-feature lifts).
- **Image-only:** just the 768 CNN features. Zero tabular info.
- **Combined:** tabular + image, side by side.

If photos carry signal that the tabular features can't, the combined model should pull ahead. If the two sources just duplicate each other, combined will tie the stronger of the two.

We also include every dog in training — including the 194 with no photos, whose CNN features are zero-padded. These dogs adopt at a much lower rate (~30% vs ~47%); including them gives the model a wider range of examples. We evaluate on the photo-having test subset, so the comparison is fair.

For the demos above we extracted features from a single photo per dog. For the model comparison we'll do something a bit better — load the multi-photo features from `m06-photos.zip`. Each row is one dog, but the 768 numbers are *averaged across all of that dog's photos* (3–4 on average, up to 30). PetFinder dogs typically have multiple photos in their listing; using all of them gives a smoother, more representative feature vector. We'll come back to this design choice after looking at the numbers.

In [ ]:
photo_df = pd.read_csv(f'{DATA_DIR}/petfinder_cnn_features.csv').set_index('PetID')
IMAGE = [f'img_{i}' for i in range(768)]
print(f'Pre-extracted multi-photo CNN features: {photo_df.shape[0]:,} dogs × {photo_df.shape[1]} features')

merged = dogs.merge(photo_df, left_on='PetID', right_index=True, how='left')
has_photo = merged[IMAGE[0]].notna().values
merged[IMAGE] = merged[IMAGE].fillna(0.0)
print(f'Dogs: {len(merged):,} total ({has_photo.sum():,} with photos, {(~has_photo).sum():,} zero-padded)')


In [ ]:
def to_shared6(d):
    out = pd.DataFrame()
    out['age_adult']      = (d['Age'] >= 12).astype(int)
    out['is_young']       = (d['Age'] < 6).astype(int)
    out['is_mixed_breed'] = ((d['Breed1'] == 307) | (d['Breed2'] != 0)).astype(int)
    out['gender_male']    = (d['Gender'] == 1).astype(int)
    out['gender_female']  = (d['Gender'] == 2).astype(int)
    out['sterilized_yes'] = (d['Sterilized'] == 1).astype(int)
    return out

X_tab  = to_shared6(merged).values.astype(float)
X_img  = merged[IMAGE].values
X_both = np.concatenate([X_tab, X_img], axis=1)
y      = merged['adopted_in_30_days'].values

tr_idx, te_idx = train_test_split(
    np.arange(len(y)), test_size=0.25, random_state=42, stratify=y,
)

# Evaluate on dogs with real photos — zero-padded dogs enrich training
# but we measure photo-model accuracy where there are real photos.
te_eval = te_idx[has_photo[te_idx]]
y_tr, y_te = y[tr_idx], y[te_eval]

def scale(X, tr, te):
    s = StandardScaler().fit(X[tr])
    return s.transform(X[tr]), s.transform(X[te])

X_tr_tab,  X_te_tab  = scale(X_tab,  tr_idx, te_eval)
X_tr_img,  X_te_img  = scale(X_img,  tr_idx, te_eval)
X_tr_both, X_te_both = scale(X_both, tr_idx, te_eval)

print(f'Train: {len(tr_idx):,} (all dogs)  |  Test: {len(te_eval):,} (photo-having)')
print(f'Feature counts:  tab={X_tab.shape[1]}  img={X_img.shape[1]}  both={X_both.shape[1]}')


In [ ]:
class TabularMLP(nn.Module):
    """Same MLP shape from 5.3 / M03 — works on any feature set."""

    def __init__(self, n_features, hidden=(64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X_tr, y_tr, X_te, y_te, epochs=150, seed=2):
    """Train an MLP, return (accuracy, AUC) on the held-out test set."""
    torch.manual_seed(seed)
    Xtr = torch.tensor(X_tr, dtype=torch.float32, device=device)
    ytr = torch.tensor(y_tr, dtype=torch.float32, device=device)
    Xte = torch.tensor(X_te, dtype=torch.float32, device=device)
    model = TabularMLP(Xtr.shape[1]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss_fn(model(Xtr), ytr).backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte)).cpu().numpy()
    preds = (probs > 0.5).astype(int)
    acc = float((preds == y_te).mean())
    auc = float(roc_auc_score(y_te, probs))
    return acc, auc

In [ ]:
results = {
    'Tabular only': train_mlp(X_tr_tab,  y_tr, X_te_tab,  y_te),
    'Image only':   train_mlp(X_tr_img,  y_tr, X_te_img,  y_te),
    'Combined':     train_mlp(X_tr_both, y_tr, X_te_both, y_te),
}

print(f'{"":14s}  {"accuracy":>9}  {"AUC":>7}')
for name, (acc, auc) in results.items():
    print(f'{name:14s}  {acc:>9.3f}  {auc:>7.3f}')

In [ ]:
names = list(results.keys())
accs  = [results[n][0] for n in names]
aucs  = [results[n][1] for n in names]

fig, (ax_acc, ax_auc) = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
colors = ['#5BC0DE', '#F0AD4E', '#5CB85C']

for ax, vals, title, ylim in [
    (ax_acc, accs, 'Accuracy', (0.55, 0.68)),
    (ax_auc, aucs, 'AUC',      (0.60, 0.72)),
]:
    bars = ax.bar(names, vals, color=colors, edgecolor='white', linewidth=2)
    ax.set_ylim(*ylim)
    ax.set_title(title, fontweight='bold')
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontweight='bold')

fig.suptitle(f'Tabular vs image vs combined (SHARED_6, {len(te_eval):,} photo-having test dogs)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### Reading the result

Three patterns worth noticing across both panels.

**Tabular and image land in the same ballpark.** The 6-feature SHARED set and the 768-dim CNN features produce similar AUC — the gap is less than 0.01. That's a very different picture from M05's 5-feature OVERLAP set, which trailed image-only by ~0.04 AUC. The difference: SHARED_6 includes `is_young` and `is_mixed_breed`, M05's two biggest single-feature lifts. With the right demographics, tabular carries comparable signal to a pretrained CNN that processes every pixel in the photo.

**Combined pulls clearly ahead.** Once you put both sources together, AUC jumps by about +0.03 over either one alone. That means the two feature sets are carrying genuinely *different* information about each dog. The CNN sees composition, lighting, what the dog is doing, the room it's in. The tabular features encode demographics the CNN can't always extract reliably — whether the dog is sterilized, its breed mix, its exact age bracket. Each fills in gaps the other can't.

**Multi-photo matters.** The bundle's CSV averaged the CNN features across every photo each listing had (3–4 on average). If we'd used only the first photo per dog, the combined model would land about two points worse on both metrics. Real listings often have a great photo and a few mediocre ones; averaging across them all smooths out the bad shots and reinforces the good ones. Same code, same model, just more photos through it.

**Including zero-photo dogs helped.** We trained on all dogs — including the 194 with no photos, whose CNN features were zero-padded. These dogs adopt at ~30% (vs ~47% for dogs with photos), so they give the model a wider spread of adoption outcomes. Every configuration gains about +0.02 AUC from this more diverse training set, even though we only measure accuracy on the photo-having test dogs.

The takeaway isn't "pick one source." It's that pretrained CNN features and good tabular features are *complementary* — neither dominates, and combining them is better than either alone. The next question is whether a smarter architecture can extract even more from the combination. That's 6.3.

## Pawpularity flags as a reference point

There's a separate Kaggle competition called *Pawpularity Score* — same dataset author, same domain, completely different framing. They asked humans to label each photo with 12 binary flags:

`Subject Focus, Eyes, Face, Near, Action, Accessory, Group, Collage, Human, Occlusion, Info, Blur`

Those flags are essentially a *manual* version of what a CNN learns to detect on its own. Each one is a property that hand-engineering could approximate, but probably worse than a pretrained CNN already does. The Pawpularity competition shows those 12 flags carry useful signal — but it took human annotators to produce them. Using a pretrained CNN gets you something similar without the labor.

We're not using the Pawpularity dataset in this course — the deep dive `M06-deep_photo_failure_modes.ipynb` mentions it as a natural follow-up if you want to dig into photo-quality scoring separately.

## Why an MLP works fine here

You might have heard that gradient-boosted trees usually win on tabular data. They sometimes do — but only when there's enough signal in the data for the differences between models to matter. On PetFinder you saw in 5.5 that even very different model classes converge near 60%. The MLP is a reasonable choice, especially because:

- It's the model class you built in M03.
- It accepts arbitrary numeric features — tabular, image, or both — without changing shape.
- The combined model in this lecture *only* needed one model class, not two stacked together.

For this course we use MLPs for tabular and a pretrained CNN for image features, both trained with the same patterns from M03.

---

## Summary

- Photos predict adoption; the question is how to use them as features.
- Hand-engineered image features (brightness, sharpness, etc.) capture the obvious and miss the rest. We didn't run them in this lecture, but you can try them in the lab — the experiment lands flat.
- Pretrained CNN features give you 768 numbers per photo that capture composition, subject, and visual quality without manual coding. You get them by replacing the model's classification head with a feature-passthrough — `nn.Identity()` for a single-Linear head, or dropping just the final `Linear` for ConvNeXt's `LayerNorm + Flatten + Linear` head.
- On the full PetFinder dog set with multi-photo averaging and all-dogs training, the 6-feature SHARED set and image-only CNN features land at similar AUC (~0.66 vs ~0.67). With the right tabular features, demographics compete with 768 CNN features.
- **The combined model — tabular + image — pulls clearly ahead** at ~0.64 accuracy / ~0.70 AUC, about +0.03 AUC over either source alone. The two sources carry complementary information.
- **Use all the photos a listing has, not just the first.** Multi-photo averaging adds ~2 accuracy points / ~0.02 AUC over single-photo features for free. Same code, more loops.
- **Include zero-photo dogs in training.** Even dogs with no photos (zero-padded CNN features) help by widening the training distribution.

### Where M06 goes from here

- **6.2 — Lab:** You build this combined model end-to-end on your assigned dog: extract features for your dog's photo (or photos), train tab + image + combined, predict your dog's adoption probability.
- **6.3–6.4:** The combined model's flat-concat architecture treats every input dimension equally. A multi-task architecture — shared CNN trunk + auxiliary head predicting `is_young` — does better by letting `is_young` work through two channels at once: tabular input AND trunk-shaping aux target. SHARED_6 + CNN + MultiTask reaches **AUC 0.727** — that's Key Moment 1 of M06, and the deployable model for Taiwan.
- **6.5–6.6:** Apply the full toolkit to Taiwan COA — the cross-dataset deployment story M05 set up but didn't finish.
- **6.7:** Responsible vision-ML — what's deployable, what isn't, and the handoff to M07's transformers.
- **6.8–6.9:** Build your Taiwan dog's profile, then defend it in the M06 capstone brief.

---

**Next session (Lab 6.2):** Same pipeline, your dog. Train the three-model comparison at scale, then run your dog's photo through it for a personal adoption-probability estimate.